# Day 042 — Exercise 2: run_query

**What you'll build:** `run_query(conn, sql, params=()) -> list[dict]` — execute any SQL statement and return the rows as a list of dicts, with column names from `cursor.description`.

**Why it matters:** `cursor.fetchall()` returns plain tuples — you have to know column positions to access values. Converting to dicts (via `cursor.description`) makes results self-documenting and indexable by name: `row['revenue']` instead of `row[6]`. This single helper replaces all boilerplate for the rest of the day.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import sqlite3
import pandas as pd


import sqlite3

def setup_db(conn):
    cur = conn.cursor()
    cur.execute('''
        CREATE TABLE IF NOT EXISTS orders (
            order_id  INTEGER PRIMARY KEY,
            product   TEXT,
            category  TEXT,
            region    TEXT,
            price     REAL,
            quantity  INTEGER,
            revenue   REAL
        )''')
    cur.execute('''
        CREATE TABLE IF NOT EXISTS products (
            product    TEXT PRIMARY KEY,
            category   TEXT,
            unit_price REAL
        )''')
    rows = [
        (1,'Widget','Electronics','North',25.0,10,250.0),
        (2,'Gadget','Electronics','South',150.0,3,450.0),
        (3,'Widget','Electronics','South',25.0,5,125.0),
        (4,'Doohickey','Accessories','East',8.0,50,400.0),
        (5,'Gadget','Electronics','East',150.0,7,1050.0),
        (6,'Widget','Electronics','East',25.0,4,100.0),
        (7,'Doohickey','Accessories','North',8.0,20,160.0),
        (8,'Gadget','Electronics','North',150.0,2,300.0),
        (9,'Widget','Electronics','West',25.0,6,150.0),
        (10,'Doohickey','Accessories','South',8.0,15,120.0),
        (11,'Thingamajig','Accessories','North',200.0,1,200.0),
        (12,'Thingamajig','Accessories','East',200.0,4,800.0),
    ]
    cur.executemany(
        'INSERT OR IGNORE INTO orders VALUES (?,?,?,?,?,?,?)', rows
    )
    products = [
        ('Widget','Electronics',25.0),
        ('Gadget','Electronics',150.0),
        ('Doohickey','Accessories',8.0),
        ('Thingamajig','Accessories',200.0),
    ]
    cur.executemany(
        'INSERT OR IGNORE INTO products VALUES (?,?,?)', products
    )
    conn.commit()


conn = sqlite3.connect(':memory:')
setup_db(conn)

## Your Implementation

In [ ]:
def run_query(conn, sql, params=()):
    """
    Execute a SQL query and return rows as a list of dicts.

    Args:
        conn   — sqlite3 connection
        sql    — SQL string (may contain ? placeholders)
        params — tuple of values for ? placeholders (default: empty)
    Returns:
        list[dict] — one dict per row, keys = column names
    """
    cur = conn.cursor()
    # TODO: cur.execute(sql, params)
    # TODO: cols = [col[0] for col in cur.description]
    # TODO: return [dict(zip(cols, row)) for row in cur.fetchall()]
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    # Check 1: function defined
    try:
        assert 'run_query' in globals()
        passed += 1; print('\u2705 Check 1: run_query is defined')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: returns a list
    try:
        result = run_query(conn, 'SELECT * FROM orders')
        assert isinstance(result, list), f'expected list, got {type(result).__name__}'
        passed += 1; print('\u2705 Check 2: returns a list')
    except Exception as e:
        print(f'\u274c Check 2: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 3: returns 12 rows
    try:
        assert len(result) == 12, f'expected 12 rows, got {len(result)}'
        passed += 1; print('\u2705 Check 3: SELECT * returns 12 rows')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: rows are dicts with column names
    try:
        row = result[0]
        assert isinstance(row, dict), f'expected dict, got {type(row).__name__}'
        assert 'revenue' in row, f'revenue column missing, got {list(row.keys())}'
        passed += 1; print(f'\u2705 Check 4: rows are dicts (keys: {list(row.keys())})')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: params= works for WHERE clause
    try:
        north = run_query(conn,
                          'SELECT * FROM orders WHERE region = ?', ('North',))
        assert len(north) == 4, f'North should have 4 rows, got {len(north)}'
        assert all(r['region'] == 'North' for r in north)
        passed += 1; print('\u2705 Check 5: parameterized WHERE works (North=4 rows)')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def run_query(conn, sql, params=()):
    cur = conn.cursor()
    cur.execute(sql, params)
    cols = [col[0] for col in cur.description]
    return [dict(zip(cols, row)) for row in cur.fetchall()]
```

</details>